# 0.5 — Auto-label Open Images with a pretrained YOLO

## Purpose

This notebook reads the **raw Open Images COCO export** that notebook 00
produced and uploaded to Drive — the exact same input notebook 01 consumes,
and **NOT** via DVC. It runs a pretrained COCO YOLO over those images to
**add missing bounding boxes** (pseudo-labeling / densification), especially
`food`, which Open Images does not annotate. The enriched COCO will later be
re-uploaded to Drive as a **new version `v2`**, leaving the original `v1`
pristine; notebook 01 will be pointed at `v2` in a later step.

What it does / does NOT do:

- Works **only** on Open Images-sourced data.
- Does **NOT** touch UEC food images or their labels.
- Does **NOT** retrain anything; it only uses a pretrained model for inference.
- Does **NOT** use DVC. The input is the raw COCO export read straight from Drive.

Known limitation: COCO has no flat `plate` class (only `bowl`), so `plate`
densification is inherently bounded by what the COCO model can detect.

## 1. Repository setup

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/LucasGVallejos/iaa-visual-table-assistant.git"
REPO_DIR = Path("/content/iaa-visual-table-assistant")

%cd /content

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already present, pulling latest changes...")
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

[Errno 2] No such file or directory: '/content'
/Users/alexanderarmua/Projects/UTN/iaa-visual-table-assistant/notebooks
fatal: could not create leading directories of '/content/iaa-visual-table-assistant': Read-only file system
[Errno 2] No such file or directory: '/content/iaa-visual-table-assistant'
/Users/alexanderarmua/Projects/UTN/iaa-visual-table-assistant/notebooks


## 2. Dependencies and GPU check

In [2]:
!pip install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [3]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "[WARN] CUDA is not available in this runtime. YOLO will run on CPU "
        "and inference will be extremely slow. "
        "Switch to a GPU runtime: Runtime > Change runtime type > GPU."
    )

CUDA available: False
[WARN] CUDA is not available in this runtime. YOLO will run on CPU and inference will be extremely slow. Switch to a GPU runtime: Runtime > Change runtime type > GPU.


## 3. Mount Google Drive

Drive holds the raw Open Images zip that notebook 00 produced.

In [5]:
from google.colab import drive
drive.mount("/content/drive")

ModuleNotFoundError: No module named 'google.colab'

## 4. Extract and inspect raw Open Images

Extracts the Open Images zip from Drive into
`datasets/raw_datasets/open_images_subset/` (reusing the same helpers
notebook 01 uses) and inspects the COCO export.

> **NOTE — Drive path mismatch.** Notebook 00 uploads to
> `raw_datasets/open_images/`, but this step (like notebook 01) reads from
> `raw_datasets/open_images_subset/`. Ensure the zip lives at the expected
> `open_images_subset/` path on Drive, or move/rename it there before running.

In [ ]:
!python -m src.data.auto_label.prepare_open_images_input --samples 3

## 5. Review a few Open Images samples (current boxes)

Confirms the raw COCO reads and renders correctly before any auto-labeling.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

from src.utils.paths import get_outputs_dir

checks_dir = get_outputs_dir() / "auto_label_checks" / "phase3_raw"
for png in sorted(checks_dir.glob("*.png")):
    display(Image(str(png)))